# XeNH Geochemical Model - Google Colab

This notebook runs Monte Carlo simulations to find validated parameter combinations for Xenon and Nitrogen isotope evolution in Earth's mantle.

**⚠️ Note:** This finds REAL validated successes (not synthetic data). Parameters are tested against actual Xe and N isotopic constraints.

## Instructions
1. Run all cells in order
2. The simulation will run for several hours
3. Results will be saved and can be downloaded
4. You can adjust the number of trials in the configuration section

## Setup: Install Dependencies

In [ ]:
!pip install numpy scipy matplotlib pandas --quiet
print("✓ Dependencies installed")

## Clone Repository

**Note:** This notebook clones from the `claude/google-colab-support-01CL1eMMoLYfXqHoiGmoNCdM` branch which contains the Python implementation.

Once this branch is merged into `main`, you can update the cell below to use:
```python
!git clone https://github.com/jkrantz159/XeNH.git  # Will use main branch
```

In [ ]:
import os
if not os.path.exists('XeNH'):
    # Clone the specific branch with Python implementation and Colab support
    !git clone -b claude/google-colab-support-01CL1eMMoLYfXqHoiGmoNCdM https://github.com/jkrantz159/XeNH.git
    print("✓ Repository cloned from branch: claude/google-colab-support-01CL1eMMoLYfXqHoiGmoNCdM")
else:
    print("✓ Repository already exists")

%cd XeNH/python/src

## Verify Setup

Check that all files were cloned correctly:

In [ ]:
# Verify the clone worked and files are in place
import os
import sys

# Add current directory to Python import path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
    print(f"✓ Added {current_dir} to sys.path")

print("\nCurrent directory:", os.getcwd())
print("\nContents of current directory:")
!ls -la

print("\n" + "="*50)
print("Checking for required Python modules:")
print("="*50)

required_files = ['model_config.py', 'model_utils.py', 'parallel_xe_model.py', 'parallel_n_model.py']
all_present = True
for file in required_files:
    exists = os.path.exists(file)
    status = "✓" if exists else "✗"
    print(f"{status} {file}: {'Found' if exists else 'MISSING'}")
    all_present = all_present and exists

if all_present:
    print("\n✓ All required files are present!")
    print("✓ Ready to run simulation!")
else:
    print("\n✗ Some files are missing. Check the clone step above.")

# Test import
print("\n" + "="*50)
print("Testing imports:")
print("="*50)
try:
    from model_config import ModelConfig
    print("✓ Successfully imported ModelConfig")
    print(f"  - Earth age: {ModelConfig.EARTH_AGE_YEARS/1e9:.3f} Gyr")
except ImportError as e:
    print(f"✗ Import failed: {e}")
    print(f"\nPython path: {sys.path[:3]}")


## Configuration

Adjust simulation parameters here:

In [ ]:
# Configuration
NUM_TRIALS = 500000  # Adjust this: 100k (~1.5 hrs), 500k (~4.6 hrs), 1M (~9 hrs)
RANDOM_SEED = 42     # For reproducibility
SAVE_EVERY = 5000    # Save progress every N trials

print(f"Configuration:")
print(f"  Trials: {NUM_TRIALS:,}")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Estimated time: ~{NUM_TRIALS/30/3600:.1f} hours at 30 iter/s")

## Run Simulation

This will run the Monte Carlo simulation. It may take several hours depending on NUM_TRIALS.

**Progress updates every 5%**

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
import time
import sys
import os

# Ensure the current directory is in Python's import path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Import model modules
from model_config import ModelConfig
from parallel_xe_model import parallel_xe_model
from parallel_n_model import parallel_n_model

print("="*70)
print("  REAL Monte Carlo Simulation - Finding Validated Successes")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print("⚠️  This finds REAL validated successes (not synthetic data)")
print("   Parameters are tested against actual Xe and N criteria\n")

print(f"Configuration:")
print(f"  Trials: {NUM_TRIALS:,}")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Expected time: ~{NUM_TRIALS/30/3600:.1f} hours at 30 iter/s")
print(f"  Expected successes: Difficult to predict (very low rate)\n")

# Set random seed
np.random.seed(RANDOM_SEED)

# Setup model
t = ModelConfig.create_time_vector()
T = ModelConfig.EARTH_AGE_YEARS
atm = ModelConfig.calculate_atmospheric_evolution(t)

print(f"Model setup:")
print(f"  Time steps: {len(t)}")
print(f"  Integration period: 0 to {T/1e9:.3f} Ga")
print(f"  Xe constraints: 130Xe=[{ModelConfig.XE130_MIN:.1e}, {ModelConfig.XE130_MAX:.1e}]")
print(f"                 128/130=[{ModelConfig.XE128_130_MIN}, {ModelConfig.XE128_130_MAX}]")
print(f"  N constraints:  14N=[{ModelConfig.N14_MIN:.1e}, {ModelConfig.N14_MAX:.1e}] mol/g")
print(f"                 15/14=[{ModelConfig.N15_14_MIN}, {ModelConfig.N15_14_MAX}]\n")

print("Starting simulation...")
print("Progress reports every 5% of completion\n")

# Initialize
successes = []
success_count = 0
start_time = time.time()
report_interval = NUM_TRIALS // 20  # Report every 5%

# Main simulation loop
for count in range(1, NUM_TRIALS + 1):
    # Generate random parameters
    alpha = 10 ** (-10 + (-7 + 10) * np.random.rand())  # 1E-10 to 1E-7
    beta = 10 * np.random.rand() * 1e9                   # 0 to 10 Gyr
    eta = ModelConfig.ETA_MIN + np.random.rand() * (ModelConfig.ETA_MAX - ModelConfig.ETA_MIN)
    resfrac = ModelConfig.RESFRAC_MIN + np.random.rand() * (ModelConfig.RESFRAC_MAX - ModelConfig.RESFRAC_MIN)
    lv_frac = 10 ** (-4 + (-2 + 4) * np.random.rand())   # 1E-4 to 1E-2
    xe_cap = 10 ** (5 + (6 - 5) * np.random.rand())      # 1E5 to 1E6
    n_cap = 10 ** (20 + (22 - 20) * np.random.rand())    # 1E20 to 1E22
    
    # Test BOTH models against actual criteria
    n_succ = parallel_n_model(n_cap, alpha, beta, eta, resfrac, lv_frac, 0, t, T, count)
    xe_succ = parallel_xe_model(xe_cap, alpha, beta, eta, resfrac, lv_frac, 0, atm, t, T, count)
    
    # Only save if BOTH succeed
    if n_succ == 1 and xe_succ == 1:
        success_count += 1
        successes.append({
            'eta': eta,
            'alpha': alpha,
            'beta': beta / 1e9,  # Convert to Gyr
            'xe_cap': xe_cap,
            'n_cap': n_cap,
            'resfrac': resfrac,
            'lv_frac': lv_frac
        })
        print(f"\n✓ SUCCESS #{success_count} found at iteration {count}!")
        sys.stdout.flush()
    
    # Progress reporting
    if count % report_interval == 0:
        elapsed = time.time() - start_time
        rate = count / elapsed
        remaining = (NUM_TRIALS - count) / rate
        percent = 100 * count / NUM_TRIALS
        print(f"Progress: {percent:5.1f}% ({count:,}/{NUM_TRIALS:,}) | "
              f"Successes: {success_count} | "
              f"Rate: {rate:.1f} iter/s | "
              f"Time remaining: {remaining/3600:.2f} hrs")
        sys.stdout.flush()

# Save results
elapsed = time.time() - start_time

print("\n" + "="*70)
print("  SIMULATION COMPLETE")
print("="*70)
print(f"Total time: {elapsed/3600:.2f} hours")
print(f"Total trials: {NUM_TRIALS:,}")
print(f"Total successes: {success_count}")
print(f"Success rate: {100*success_count/NUM_TRIALS:.4f}%")
print(f"Average rate: {NUM_TRIALS/elapsed:.1f} iterations/second\n")

if success_count > 0:
    df = pd.DataFrame(successes)
    output_file = '../results/success_REAL_colab.csv'
    df.to_csv(output_file, index=False)
    print(f"✓ Results saved to: {output_file}")
    print(f"\nFirst few successes:")
    print(df.head())
else:
    print("⚠️  No validated successes found.")
    print("   Try increasing NUM_TRIALS or adjusting parameter ranges.")

## Visualize Results

If successes were found, create visualization plots:

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

if success_count > 0:
    df = pd.read_csv('../results/success_REAL_colab.csv')
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'Validated Parameter Distributions ({success_count} successes)', fontsize=16)
    
    # Processing rate (eta)
    ax = axes[0, 0]
    ax.hist(df['eta'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Processing rate η')
    ax.set_ylabel('Count')
    ax.set_title('Processing Rate Distribution')
    ax.grid(alpha=0.3)
    
    # Growth rate (alpha)
    ax = axes[0, 1]
    ax.hist(np.log10(df['alpha']), bins=30, color='coral', alpha=0.7, edgecolor='black')
    ax.set_xlabel('log₁₀(α)')
    ax.set_ylabel('Count')
    ax.set_title('Growth Rate Distribution')
    ax.grid(alpha=0.3)
    
    # Inflection point (beta)
    ax = axes[0, 2]
    ax.hist(df['beta'], bins=30, color='seagreen', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Inflection point β (Gyr)')
    ax.set_ylabel('Count')
    ax.set_title('Inflection Point Distribution')
    ax.grid(alpha=0.3)
    
    # Xe capacity
    ax = axes[1, 0]
    ax.hist(np.log10(df['xe_cap']), bins=30, color='purple', alpha=0.7, edgecolor='black')
    ax.set_xlabel('log₁₀(Xe capacity)')
    ax.set_ylabel('Count')
    ax.set_title('Xe Capacity Distribution')
    ax.grid(alpha=0.3)
    
    # N capacity
    ax = axes[1, 1]
    ax.hist(np.log10(df['n_cap']), bins=30, color='orange', alpha=0.7, edgecolor='black')
    ax.set_xlabel('log₁₀(N capacity)')
    ax.set_ylabel('Count')
    ax.set_title('N Capacity Distribution')
    ax.grid(alpha=0.3)
    
    # Recycling fraction
    ax = axes[1, 2]
    ax.hist(df['resfrac'], bins=30, color='crimson', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Recycling fraction')
    ax.set_ylabel('Count')
    ax.set_title('Recycling Fraction Distribution')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/parameter_distributions_colab.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("\n✓ Visualization saved to: ../results/parameter_distributions_colab.png")
    
    # Summary statistics
    print("\nSummary Statistics:")
    print(df.describe())
else:
    print("No results to visualize.")

## Download Results

Download the results file to your local machine:

In [ ]:
from google.colab import files

if success_count > 0:
    # Download CSV
    files.download('../results/success_REAL_colab.csv')
    print("✓ CSV downloaded")
    
    # Download figure if it exists
    if os.path.exists('../results/parameter_distributions_colab.png'):
        files.download('../results/parameter_distributions_colab.png')
        print("✓ Figure downloaded")
else:
    print("No results to download.")

## Notes

### Understanding the Results

The simulation finds parameter combinations where **BOTH** Xenon and Nitrogen isotope models satisfy observational constraints:

**Xenon constraints:**
- Final ¹³⁰Xe concentration: 4.3×10⁵ - 9.2×10⁵
- Final ¹²⁸Xe/¹³⁰Xe ratio: 0.475 - 0.478

**Nitrogen constraints:**
- Final ¹⁴N concentration: 7.1×10¹⁹ - 9.8×10²¹ mol/g
- Final ¹⁵N/¹⁴N ratio: 0.0036275 - 0.0036425

### Parameters

- **η (eta)**: Processing rate parameter
- **α (alpha)**: Growth rate for sigmoidal downwelling (1E-10 to 1E-7)
- **β (beta)**: Inflection point for downwelling onset (0-10 Gyr)
- **xe_cap**: Xenon reservoir capacity (1E5 to 1E6)
- **n_cap**: Nitrogen reservoir capacity (1E20 to 1E22)
- **resfrac**: Recycling fraction
- **lv_frac**: Late veneer fraction (1E-4 to 1E-2)

### Citation

If you use this code in published research, please cite the XeNH repository:
https://github.com/jkrantz159/XeNH